<a href="https://colab.research.google.com/github/AliGulzar-ml/dsa405-project/blob/main/Project/DSA405_002_FA26_P3_agulzar2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# P3: Acquisition & Integration

**Name:** Ali Gulzar  
**Unity ID:** agulzar2  
**Course:** DSA 405-002, Fall 2026  
**Project:** Temperature near Cushing and WTI crude-oil price changes

**Research question:** Is there an association between daily average temperature near Cushing, Oklahoma, and daily percentage changes in the Cushing WTI crude-oil spot price from 2015 through 2025?

P3 acquires and caches the EIA web source, reconstructs the audited NOAA table from P2, joins the sources by date, and verifies that the integration behaves as designed.

## Setup

In [1]:
from pathlib import Path
import os
import subprocess

REPO_DIR = Path("/content/dsa405-project")
PROJECT_DIR = REPO_DIR / "Project"

# Clone the repository only if it has not already been downloaded.
if not REPO_DIR.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/AliGulzar-ml/dsa405-project.git",
            str(REPO_DIR),
        ],
        check=True,
    )

# Make the Project folder the notebook's working directory.
os.chdir(PROJECT_DIR)

print("Working directory:", Path.cwd())
print("Raw-data folder exists:", Path("data/raw").exists())

Working directory: /content/dsa405-project/Project
Raw-data folder exists: True


## 1. Acquisition Code

The web source for P3 is the U.S. Energy Information Administration's daily Cushing WTI spot-price series (`RWTC`). The acquisition code below:

- sends an honest coursework `User-Agent`;
- waits at least one second before each network request;
- uses a 30-second timeout;
- checks non-200 responses and connection failures;
- reads the live rate limit from `X-RateLimit-Limit` and `X-RateLimit-Remaining` response headers;
- supports pagination through `offset` and `length`;
- saves each unmodified JSON response in `data/raw/` and uses it on later calls;
- validates that the expected response path and fields are present; and
- raises a clear error instead of silently continuing with invalid data.

Pandas and NumPy handle the data. The remaining imports are Python standard-library modules required for HTTP controls, timing, JSON, and caching; no additional package is required.

In [2]:
import json
import time
import urllib.parse
import urllib.request
from urllib.error import HTTPError, URLError
from pathlib import Path

import numpy as np
import pandas as pd

RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

EIA_ENDPOINT = "https://api.eia.gov/v2/petroleum/pri/spt/data/"
API_KEY = "DEMO_KEY"
USER_AGENT = "NCSU DSA 405 coursework; Ali Gulzar; educational data project"
START_DATE = "2015-01-01"
END_DATE = "2025-12-31"
PAGE_SIZE = 5000
REQUEST_DELAY_SECONDS = 1.0
TIMEOUT_SECONDS = 30


def build_eia_url(offset):
    params = [
        ("api_key", API_KEY),
        ("frequency", "daily"),
        ("data[0]", "value"),
        ("facets[series][]", "RWTC"),
        ("start", START_DATE),
        ("end", END_DATE),
        ("sort[0][column]", "period"),
        ("sort[0][direction]", "asc"),
        ("offset", str(offset)),
        ("length", str(PAGE_SIZE)),
    ]
    return EIA_ENDPOINT + "?" + urllib.parse.urlencode(params)


def require_success_status(status):
    if status != 200:
        raise RuntimeError(f"EIA request failed with HTTP status {status}")


def validate_eia_payload(payload):
    if not isinstance(payload, dict):
        raise TypeError("EIA response must be a JSON object")

    response = payload.get("response")
    if not isinstance(response, dict):
        raise ValueError("EIA JSON is missing the 'response' object")

    rows = response.get("data")
    if not isinstance(rows, list) or len(rows) == 0:
        raise ValueError("EIA selector response.data matched no rows")

    required_fields = {"period", "series", "value", "units"}
    for row_number, row in enumerate(rows):
        missing_fields = required_fields.difference(row)
        if missing_fields:
            raise ValueError(
                f"EIA row {row_number} is missing fields: {sorted(missing_fields)}"
            )
        if row["series"] != "RWTC":
            raise ValueError(f"Unexpected EIA series {row['series']!r}")

    try:
        total = int(response["total"])
    except (KeyError, TypeError, ValueError) as error:
        raise ValueError("EIA response has no valid total row count") from error

    if total < len(rows):
        raise ValueError("EIA total is smaller than the returned page")
    return total


def page_cache_paths(offset):
    stem = f"eia_wti_page_{offset:05d}"
    return RAW_DIR / f"{stem}.json", RAW_DIR / f"{stem}_headers.json"


def load_or_fetch_eia_page(offset):
    page_path, header_path = page_cache_paths(offset)

    if page_path.exists() != header_path.exists():
        raise RuntimeError(
            f"Incomplete cache for offset {offset}: response and header files must both exist"
        )

    if page_path.exists():
        try:
            payload = json.loads(page_path.read_text(encoding="utf-8"))
            metadata = json.loads(header_path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError) as error:
            raise RuntimeError(f"Cached EIA page {offset} is unreadable") from error

        validate_eia_payload(payload)
        if not metadata.get("x-ratelimit-limit"):
            raise RuntimeError("Cached EIA headers are missing X-RateLimit-Limit")
        return payload, metadata, "cache"

    # One request per second protects the service if more than one page is needed.
    time.sleep(REQUEST_DELAY_SECONDS)
    request = urllib.request.Request(
        build_eia_url(offset),
        headers={"User-Agent": USER_AGENT},
    )

    try:
        with urllib.request.urlopen(request, timeout=TIMEOUT_SECONDS) as response:
            status = response.status
            raw_bytes = response.read()
            metadata = {
                "http_status": status,
                "x-ratelimit-limit": response.headers.get("X-RateLimit-Limit"),
                "x-ratelimit-remaining": response.headers.get("X-RateLimit-Remaining"),
                "response_date": response.headers.get("Date"),
            }
    except HTTPError as error:
        raise RuntimeError(f"EIA request failed with HTTP status {error.code}") from error
    except (URLError, TimeoutError) as error:
        raise RuntimeError(f"EIA request failed or timed out: {error}") from error

    require_success_status(status)
    if not metadata["x-ratelimit-limit"]:
        raise RuntimeError("Live EIA response is missing X-RateLimit-Limit")

    try:
        payload = json.loads(raw_bytes.decode("utf-8"))
    except (UnicodeDecodeError, json.JSONDecodeError) as error:
        raise RuntimeError("EIA response was not valid UTF-8 JSON") from error

    validate_eia_payload(payload)

    # Cache only when the response has passed every validation above.
    page_path.write_bytes(raw_bytes)
    header_path.write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
    return payload, metadata, "network"


def acquire_eia_wti():
    all_rows = []
    acquisition_records = []
    offset = 0
    expected_total = None

    while expected_total is None or len(all_rows) < expected_total:
        payload, metadata, source = load_or_fetch_eia_page(offset)
        total = validate_eia_payload(payload)
        page_rows = payload["response"]["data"]

        if expected_total is None:
            expected_total = total
        elif total != expected_total:
            raise RuntimeError("EIA total changed between pagination pages")

        all_rows.extend(page_rows)
        acquisition_records.append(
            {
                "offset": offset,
                "rows": len(page_rows),
                "total": total,
                "source": source,
                "http_status": metadata["http_status"],
                "rate_limit": metadata["x-ratelimit-limit"],
                "rate_remaining": metadata["x-ratelimit-remaining"],
            }
        )

        if len(all_rows) >= expected_total:
            break
        if len(page_rows) == 0:
            raise RuntimeError("Pagination stopped before the advertised total")
        offset += len(page_rows)

    if len(all_rows) != expected_total:
        raise RuntimeError(
            f"Pagination returned {len(all_rows)} rows; expected {expected_total}"
        )

    return pd.DataFrame(all_rows), pd.DataFrame(acquisition_records)

In [3]:
eia_raw, acquisition_log = acquire_eia_wti()

print("Acquisition result:")
print(acquisition_log.to_string(index=False))
print(f"Rows acquired: {len(eia_raw):,}")
print(f"Pages required: {len(acquisition_log):,}")
print(f"User-Agent: {USER_AGENT}")
print(f"Delay before network requests: {REQUEST_DELAY_SECONDS:.1f} second")

# A second call in the same run must read the cache and make no new request.
eia_raw_again, second_call_log = acquire_eia_wti()
assert second_call_log["source"].eq("cache").all()
assert len(eia_raw_again) == len(eia_raw)
print("Second acquisition call used cached pages only: True")

Acquisition result:
 offset  rows  total source  http_status rate_limit rate_remaining
      0  2760   2760  cache          200         10              8
Rows acquired: 2,760
Pages required: 1
User-Agent: NCSU DSA 405 coursework; Ali Gulzar; educational data project
Delay before network requests: 1.0 second
Second acquisition call used cached pages only: True


### Demonstrated failure handling

The next cell deliberately supplies a failed HTTP status and a response whose data selector matches no rows. Both failures are expected, caught, and printed. The cell raises its own error if either bad input is allowed to pass silently.

In [4]:
caught_failures = []

try:
    require_success_status(503)
except RuntimeError as error:
    caught_failures.append(str(error))
    print("EXPECTED HTTP FAILURE CAUGHT:", error)

try:
    validate_eia_payload({"response": {"total": "0", "data": []}})
except ValueError as error:
    caught_failures.append(str(error))
    print("EXPECTED EMPTY-SELECTOR FAILURE CAUGHT:", error)

if len(caught_failures) != 2:
    raise AssertionError("Failure demonstration did not catch both invalid conditions")
print("Failure demonstration passed: both invalid conditions stopped loudly.")

EXPECTED HTTP FAILURE CAUGHT: EIA request failed with HTTP status 503
EXPECTED EMPTY-SELECTOR FAILURE CAUGHT: EIA selector response.data matched no rows
Failure demonstration passed: both invalid conditions stopped loudly.


## 2. Ethics and Legality in Practice

### Match to the P1/A8 plan and drift

The final implementation still follows the project plan: combine public EIA Cushing WTI prices with NOAA temperature observations for 2015–2025 using the calendar date. There is no drift in the question, time period, or primary sources.

There are two implementation changes worth documenting. First, the original Cushing NOAA station stopped reporting before 2015, so P2 established Stillwater 2 W, about 20 miles away, as the nearest complete temperature proxy. P3 preserves that audited choice. Second, the EIA source is collected as cached JSON through its documented API rather than relying on the downloadable spreadsheet. The API exposes status and rate-limit headers and supports reliable pagination, which directly satisfies P3's acquisition requirements.

### Guardrails reconfirmed

- Both sources are public government data sources with documented access methods.
- No login, paywall, personal data, or information about individuals is involved.
- The code does not bypass authentication, access controls, robots rules, or technical protections.
- The request identifies itself as NCSU coursework, waits before network calls, checks the live rate-limit headers, and caches successful responses.
- Raw NOAA data are never overwritten. Each EIA page is written once only after validation and is read from cache thereafter.
- EIA is credited as the source of the WTI series, and NOAA/NCEI is credited as the weather source.

## 3. Prepare the Two Tables

The NOAA raw file was preserved and audited in P2. P3 repeats the documented cleaning rules so the integration notebook runs independently from the raw source.

In [5]:
NOAA_RAW_PATH = RAW_DIR / "noaa_stillwater_daily_2015_2025.csv"
if not NOAA_RAW_PATH.exists():
    raise FileNotFoundError(
        "Missing preserved NOAA file. Run the notebook from the project repository root."
    )

weather_raw = pd.read_csv(NOAA_RAW_PATH, low_memory=False)
required_weather_columns = {"STATION", "DATE", "TMIN", "TMAX"}
missing_weather_columns = required_weather_columns.difference(weather_raw.columns)
if missing_weather_columns:
    raise ValueError(f"NOAA raw file is missing: {sorted(missing_weather_columns)}")

weather = weather_raw[["STATION", "DATE", "TMIN", "TMAX"]].copy()
weather = weather.rename(
    columns={
        "STATION": "station_id",
        "DATE": "date",
        "TMIN": "temp_min_f",
        "TMAX": "temp_max_f",
    }
)
weather["station_id"] = weather["station_id"].astype("string")
weather["date"] = pd.to_datetime(weather["date"], errors="raise")
weather["temperature_complete"] = weather[["temp_min_f", "temp_max_f"]].notna().all(axis=1)
weather["temp_avg_f"] = (weather["temp_min_f"] + weather["temp_max_f"]) / 2
weather["temp_range_f"] = weather["temp_max_f"] - weather["temp_min_f"]
weather = weather.sort_values("date", kind="stable").reset_index(drop=True)

assert weather["date"].is_unique
assert weather["station_id"].eq("USW00053926").all()
assert not (weather["temp_max_f"] < weather["temp_min_f"]).any()

print(f"Weather table: {len(weather):,} rows × {weather.shape[1]:,} columns")
print(f"Weather date range: {weather['date'].min().date()} through {weather['date'].max().date()}")
print(f"Complete temperature rows: {weather['temperature_complete'].sum():,}")

Weather table: 4,013 rows × 7 columns
Weather date range: 2015-01-01 through 2025-12-31
Complete temperature rows: 3,991


In [6]:
required_eia_columns = {"period", "series", "value", "units"}
missing_eia_columns = required_eia_columns.difference(eia_raw.columns)
if missing_eia_columns:
    raise ValueError(f"EIA data are missing: {sorted(missing_eia_columns)}")

oil = eia_raw[["period", "series", "value", "units"]].copy()
oil = oil.rename(
    columns={
        "period": "date",
        "value": "wti_price_usd_per_barrel",
    }
)
oil["date"] = pd.to_datetime(oil["date"], errors="raise")
oil["wti_price_usd_per_barrel"] = pd.to_numeric(
    oil["wti_price_usd_per_barrel"], errors="raise"
)
oil = oil.sort_values("date", kind="stable").reset_index(drop=True)
oil["daily_return_pct"] = (
    oil["wti_price_usd_per_barrel"].pct_change(fill_method=None) * 100
)

assert oil["date"].is_unique
assert oil["series"].eq("RWTC").all()

print(f"Oil table: {len(oil):,} rows × {oil.shape[1]:,} columns")
print(f"Oil date range: {oil['date'].min().date()} through {oil['date'].max().date()}")
print(oil.head().to_string(index=False))

Oil table: 2,760 rows × 5 columns
Oil date range: 2015-01-02 through 2025-12-31
      date series  wti_price_usd_per_barrel units  daily_return_pct
2015-01-02   RWTC                     52.72 $/BBL               NaN
2015-01-05   RWTC                     50.05 $/BBL         -5.064492
2015-01-06   RWTC                     47.98 $/BBL         -4.135864
2015-01-07   RWTC                     48.69 $/BBL          1.479783
2015-01-08   RWTC                     48.80 $/BBL          0.225919


## 4. Join Design — Written Before the Merge

| Design item | Decision |
|---|---|
| **Join key** | `date` |
| **Grain of the left table** | One row per WTI trading date |
| **Grain of the right table** | One row per calendar date at NOAA station `USW00053926` |
| **Relationship** | One-to-one: both tables require unique dates |
| **Expected rows after join** | **2,760**, exactly the number of oil observations |
| **Join type and why** | Left join from oil to weather so every WTI observation remains visible, even when weather is absent |
| **Unmatched oil rows** | Retained with missing weather fields and marked `left_only` |
| **Weather-only rows** | Not added because weekends and nontrading dates are outside the price-change grain |

An inner join was ruled out because it would silently hide oil dates without weather—the Week 6 **wreck** failure. An outer join was ruled out because it would add thousands of nontrading weather dates that cannot answer the price-change question. `validate="one_to_one"` and uniqueness checks guard against a **fan-out** that would multiply rows. The left join, row-count assertion, and key-set assertion guard against a wreck that would remove oil dates.

In [7]:
EXPECTED_JOIN_ROWS = 2760
EXPECTED_LEFT_ONLY = 3

print(f"Left-table rows: {len(oil):,}")
print(f"Left-table unique dates: {oil['date'].nunique():,}")
print(f"Right-table rows: {len(weather):,}")
print(f"Right-table unique dates: {weather['date'].nunique():,}")
print(f"Expected joined rows: {EXPECTED_JOIN_ROWS:,}")

if len(oil) != EXPECTED_JOIN_ROWS:
    raise AssertionError(
        f"Oil row count changed: {len(oil)} != expected {EXPECTED_JOIN_ROWS}"
    )

Left-table rows: 2,760
Left-table unique dates: 2,760
Right-table rows: 4,013
Right-table unique dates: 4,013
Expected joined rows: 2,760


In [8]:
joined = oil.merge(
    weather,
    on="date",
    how="left",
    validate="one_to_one",
    indicator=True,
)

join_counts = (
    joined["_merge"]
    .value_counts()
    .reindex(["both", "left_only", "right_only"], fill_value=0)
)

print(f"Joined rows: {len(joined):,}")
print("Join outcomes:")
print(join_counts.to_string())
print("\nOil dates without a NOAA row:")
print(
    joined.loc[
        joined["_merge"].eq("left_only"),
        ["date", "wti_price_usd_per_barrel"],
    ].to_string(index=False)
)

Joined rows: 2,760
Join outcomes:
_merge
both          2757
left_only        3
right_only       0

Oil dates without a NOAA row:
      date  wti_price_usd_per_barrel
2020-12-10                     46.76
2022-10-17                     86.00
2024-10-14                     76.11


## 5. Verification Suite

Every assertion below states the assumption it protects. If any assumption becomes false, the notebook stops instead of producing an untrustworthy integrated table.

In [9]:
# Protects against: duplicate oil keys creating or hiding join rows.
assert oil["date"].is_unique, "Oil dates must be unique"

# Protects against: duplicate weather keys causing a one-to-many fan-out.
assert weather["date"].is_unique, "Weather dates must be unique"

# Protects against: a fan-out or wreck changing the left-table observation count.
assert len(joined) == EXPECTED_JOIN_ROWS, "Left join must preserve all oil rows"

# Protects against: duplicated joined observations after integration.
assert joined["date"].is_unique, "Joined dates must remain unique"

# Protects against: an inner-join-style wreck silently removing oil dates.
assert set(joined["date"]) == set(oil["date"]), "Every oil date must survive"

# Protects against: unexpected referential-integrity drift in the NOAA coverage.
assert int(join_counts["left_only"]) == EXPECTED_LEFT_ONLY, "Unexpected unmatched oil-date count"

# Protects against: parsing errors or impossible WTI values.
assert joined["wti_price_usd_per_barrel"].between(-100, 200).all(), "WTI value outside validation bounds"

# Protects against: unit mistakes or impossible derived temperature values.
observed_temperatures = joined["temp_avg_f"].dropna()
assert observed_temperatures.between(-100, 140).all(), "Temperature outside validation bounds"

# Protects against: infinities introduced by numeric conversion or return calculation.
finite_returns = joined["daily_return_pct"].dropna()
assert np.isfinite(finite_returns).all(), "Daily returns must be finite"

print("Verification suite passed: 9 assertions.")

Verification suite passed: 9 assertions.


In [10]:
analysis_ready = joined.loc[
    joined["_merge"].eq("both")
    & joined["daily_return_pct"].notna()
    & joined["temp_avg_f"].notna(),
    [
        "date",
        "wti_price_usd_per_barrel",
        "daily_return_pct",
        "station_id",
        "temp_min_f",
        "temp_max_f",
        "temp_avg_f",
        "temp_range_f",
    ],
].copy()

assert analysis_ready["date"].is_unique
assert analysis_ready.notna().all().all()

PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_PATH = PROCESSED_DIR / "temperature_wti_join_2015_2025.csv"
analysis_ready.to_csv(PROCESSED_PATH, index=False)

print(f"Analysis-ready rows: {len(analysis_ready):,}")
print(f"Rows excluded after the verified left join: {len(joined) - len(analysis_ready):,}")
print(f"Saved processed table: {PROCESSED_PATH}")
print(analysis_ready.head().to_string(index=False))

Analysis-ready rows: 2,741
Rows excluded after the verified left join: 19
Saved processed table: data/processed/temperature_wti_join_2015_2025.csv
      date  wti_price_usd_per_barrel  daily_return_pct  station_id  temp_min_f  temp_max_f  temp_avg_f  temp_range_f
2015-01-05                     50.05         -5.064492 USW00053926        14.0        43.0        28.5          29.0
2015-01-06                     47.98         -4.135864 USW00053926        23.0        41.0        32.0          18.0
2015-01-07                     48.69          1.479783 USW00053926         9.0        29.0        19.0          20.0
2015-01-08                     48.80          0.225919 USW00053926         7.0        37.0        22.0          30.0
2015-01-09                     48.35         -0.922131 USW00053926        15.0        33.0        24.0          18.0


## One Thing That Surprised Me

The EIA series matched a NOAA date for 2,757 of 2,760 oil observations, a match rate of about 99.9%, but three valid WTI trading dates had no NOAA row at all: December 10, 2020; October 17, 2022; and October 14, 2024. Fifteen additional matched trading dates had an incomplete minimum or maximum temperature, and the first oil observation has no prior trading day from which to calculate a return. Therefore, the verified left join preserves all 2,760 oil observations, while the final complete-case table contains 2,741 rows. Seeing the missing dates explicitly is why the left join was preferable to an inner join: an inner join would have hidden the three absent weather dates before they could be investigated.

## Reproducibility Summary

- The notebook runs from top to bottom from the repository root.
- NOAA raw data remain unmodified in `data/raw/`.
- EIA responses and rate-limit headers are cached in `data/raw/` and never fetched twice in one run.
- The processed integration table is written to `data/processed/`.
- `README.md` and `data/raw/SOURCES.md` document the project and sources.
- Failure handling, join cardinality, and verification assertions are visible in the notebook.

In [11]:
print("Cached EIA response files:", len(list(RAW_DIR.glob("eia_wti_page_[0-9][0-9][0-9][0-9][0-9].json"))))
print("Cached EIA header files:", len(list(RAW_DIR.glob("eia_wti_page_*_headers.json"))))
print("NOAA raw file preserved:", NOAA_RAW_PATH.exists())
print("Processed joined file created:", PROCESSED_PATH.exists())

Cached EIA response files: 1
Cached EIA header files: 1
NOAA raw file preserved: True
Processed joined file created: True


## Self-Scored P3 Rubric

| Criterion | Weight | Self-score | Evidence |
|---|---:|---:|---|
| Acquisition code | ×2 | **3/3** | Honest User-Agent, one-second delay, timeout and HTTP handling, live rate-limit headers, pagination, validated raw cache, second-call cache proof, and two demonstrated failures. |
| Ethics and legality in practice | ×2 | **3/3** | Plan match, station/API drift, and every legal and ethical guardrail are explicitly documented. |
| Join design | ×2 | **3/3** | Key, grains, one-to-one relationship, exact expected count, left-join rationale, unmatched behavior, and rejected alternatives are written before the merge; `validate="one_to_one"` is used. |
| Verification suite | ×3 | **3/3** | Nine commented assertions check uniqueness, cardinality, row preservation, key preservation, match count, value ranges, and finite returns. |
| Reproducibility and readability | ×1 | **3/3** | The notebook, cache, processed output, README, and source record are complete. |

**Weighted self-score:** `(3×2) + (3×2) + (3×2) + (3×3) + (3×1) = 30/30`.

### Final checklist

- [x] Web/API source retrieved by working acquisition code.
- [x] User-Agent, rate limit, timeout, status, pagination, cache, and loud failure handling included.
- [x] One failure demonstrated and caught.
- [x] Ethics, legality, and drift documented.
- [x] Join design written before the merge.
- [x] Merge uses `validate="one_to_one"`.
- [x] Fan-out and wreck risks both addressed.
- [x] At least three meaningful assertions included (nine provided).
- [x] One surprising finding reported.
- [x] README and source record updated.
- [x] Upload the package to GitHub and submit the repository link.

## AI Use Note
AI was used to Format the notebook and assist with the writing of the code.